# Clasificación de textos en esperanto: física, biología y química

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danielaagonzalezz/cienciaNLP/blob/main/notebooks/clasificacion_esperanto.ipynb)

**Minería de Textos · 4º INF-A · CUNEF · 2026/2027**

**Equipo:** Daniela González ([@danielaagonzalezz](https://github.com/danielaagonzalezz)) · Olivia ([@Olivia029](https://github.com/Olivia029))

**Repositorio:** https://github.com/danielaagonzalezz/cienciaNLP

---

Este cuaderno es la **Fase 3** de la práctica. Usa lo que hicimos en las dos primeras, que está en el repositorio:

1. **Dataset** (`dataset/dataset.csv`): párrafos de la Wikipedia en esperanto etiquetados como `fiziko`, `biologio` o
   `kemio`. Lo genera el script `scripts/build_dataset.py`.
2. **Librería `ciencianlp`**: tokenizador, lematizador, POS-tagger y stop-words, basada en reglas y hecha desde cero.

Con ellos el cuaderno **vectoriza el corpus, entrena un clasificador, lo evalúa y clasifica páginas web reales**.

Hemos elegido física, biología y química porque son **temáticas cercanas**: comparten vocabulario científico
(*energio*, *molekulo*, *ĉelo*, *atomo*...), así que separarlas no es trivial.

**Independencia de temática:** el cuaderno no tiene las clases escritas en ningún sitio, las lee del dataset. Basta
con cambiar `URL_DATASET` por el dataset de otro equipo que cumpla el contrato de datos para reentrenarlo y evaluarlo.

## 0. Configuración

Todos los parámetros están en esta celda, para que el cuaderno se ejecute sin preguntar nada al usuario.

- `URL_DATASET`: dataset que se usa. Por defecto, el nuestro; se puede poner el de otro equipo.
- `URL_DATASET_RESPALDO`: copia del mismo fichero en otro servidor, por si el primero no responde.
- `SEMILLA`: se fija para que los resultados sean reproducibles.

In [ ]:
REPO = "danielaagonzalezz/cienciaNLP"
RAMA = "main"  # rama de la que se instala la librería y se lee el dataset

URL_DATASET = f"https://raw.githubusercontent.com/{REPO}/{RAMA}/dataset/dataset.csv"
URL_DATASET_RESPALDO = f"https://cdn.jsdelivr.net/gh/{REPO}@{RAMA}/dataset/dataset.csv"  # None si no hay copia

SEMILLA = 42

### Instalación

En Colab ya vienen `pandas`, `scikit-learn`, `matplotlib`, `requests` y `beautifulsoup4`. Solo falta nuestra
librería, que se instala con `pip` desde GitHub. Si ya está instalada, no se reinstala.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ciencianlp") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"git+https://github.com/{REPO}.git@{RAMA}"], check=True)

In [ ]:
import random
import re

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import sklearn

import ciencianlp

random.seed(SEMILLA)
np.random.seed(SEMILLA)

# Versiones usadas, para poder reproducir los resultados
for libreria in (ciencianlp, pd, np, sklearn, matplotlib, requests):
    print(f"{libreria.__name__:<12} {libreria.__version__}")

## 1. Dataset

### 1.1 Descarga

El dataset se descarga automáticamente del repositorio de GitHub. Para que el cuaderno no falle si GitHub no responde,
si la primera URL falla se prueba la copia del CDN jsDelivr, que sirve el mismo fichero del mismo repositorio.

In [ ]:
def descargar_dataset(*urls):
    """Lee el CSV de la primera URL que responda. Si ninguna responde, lanza un error claro."""
    for url in filter(None, urls):  # se salta las URL vacías (None)
        try:
            datos = pd.read_csv(url, sep=";", encoding="utf-8")
            print("Dataset leído de", url)
            return datos
        except Exception as error:  # sin red, 404, fichero corrupto...
            print("No se pudo leer", url, "->", error)
    raise RuntimeError("No se pudo descargar el dataset de ninguna fuente")


df = descargar_dataset(URL_DATASET, URL_DATASET_RESPALDO)
CLASES = sorted(df["class"].unique())  # las clases salen del dataset, no se escriben a mano
print("Clases:", CLASES)
df.head()

### 1.2 Comprobación del contrato de datos

El enunciado fija un contrato para el dataset. Lo comprobamos con `assert`, de forma que si algo no se cumple el
cuaderno se para y avisa:

- dos columnas con cabecera `text;class`,
- ningún párrafo vacío ni con el carácter `;`,
- al menos 3 clases,
- párrafos limpios: sin HTML, enlaces ni referencias del tipo `[1]`.

In [ ]:
assert list(df.columns) == ["text", "class"], "Las columnas deben ser text;class"
assert df["text"].notna().all() and (df["text"].str.strip() != "").all(), "Hay párrafos vacíos"
assert not df["text"].str.contains(";").any(), "Algún párrafo contiene ';'"
assert df["class"].nunique() >= 3, "Hacen falta al menos 3 clases"

restos = re.compile(r"<[a-z/][^>]*>|https?://|www\.|\[\d+\]")  # HTML, enlaces y referencias
assert not df["text"].str.contains(restos).any(), "Quedan restos de HTML, enlaces o referencias"

print(f"Contrato cumplido: {len(df)} párrafos, {df['class'].nunique()} clases")
df["class"].value_counts().rename("párrafos").to_frame()

## 2. Procesado con `ciencianlp`

Nuestra librería tiene una API parecida a la de spaCy: `nlp(texto)` devuelve un `Doc`, que se recorre token a token.
Cada `Token` trae su lema (`lemma_`), su etiqueta gramatical (`pos_`) y si es una stop-word (`is_stop`).

In [ ]:
from ciencianlp import nlp

ejemplo = nlp("La elektronoj de la atomo moviĝas ĉirkaŭ la kerno.")
pd.DataFrame([{"text": t.text, "lemma_": t.lemma_, "pos_": t.pos_, "is_stop": t.is_stop, "is_punct": t.is_punct}
              for t in ejemplo])

### 2.1 Preprocesado del corpus

De cada párrafo nos quedamos con los **lemas** de las palabras que aportan información sobre el tema:

- **Lematizar** junta las formas de una misma palabra: *ĉelo*, *ĉeloj*, *ĉelon* y *ĉelojn* cuentan como `ĉelo`. Así
  hay menos columnas en la tabla y cada una tiene más ejemplos.
- **Quitar stop-words** (*la*, *de*, *kaj*, *estas*...), porque aparecen igual en las tres clases.
- **Quitar signos y números** (`pos_ == "NUM"`): un año o una cifra no indica la temática.

Guardamos también las palabras sin lematizar (`palabras`), para comparar más adelante si la librería mejora el modelo.

In [ ]:
import time


def lemas_utiles(doc):
    """Lemas que aportan información temática: sin stop-words, signos ni números."""
    return [t.lemma_ for t in doc
            if not (t.is_stop or t.is_punct or t.pos_ == "NUM") and len(t.text) > 1]


inicio = time.perf_counter()
docs = [nlp(texto) for texto in df["text"]]
df["lemas"] = [lemas_utiles(doc) for doc in docs]
df["palabras"] = [[t.text.lower() for t in doc if not t.is_punct] for doc in docs]
print(f"{len(docs)} párrafos procesados en {time.perf_counter() - inicio:.1f} s")

formas = {t.text.lower() for doc in docs for t in doc if not t.is_punct}
lemas = {t.lemma_ for doc in docs for t in doc if not t.is_punct}
print(f"Vocabulario: {len(formas)} formas distintas -> {len(lemas)} lemas "
      f"({1 - len(lemas) / len(formas):.0%} menos gracias al lematizador)")

### 2.2 Etiquetas gramaticales del corpus

Porcentaje de tokens de cada categoría gramatical en cada clase. La fila `X` son las palabras que el etiquetador no
sabe clasificar (nombres propios sin terminación esperanta, siglas, fórmulas...); sirve para ver hasta dónde llegan
las reglas.

In [ ]:
etiquetas = pd.DataFrame([(clase, t.pos_) for doc, clase in zip(docs, df["class"]) for t in doc],
                         columns=["class", "pos"])
tabla_pos = pd.crosstab(etiquetas["pos"], etiquetas["class"], normalize="columns").mul(100).round(1)
tabla_pos.sort_values(tabla_pos.columns[0], ascending=False)

## 3. Exploración del dataset

### 3.1 Tamaño de las clases y longitud de los párrafos

Comprobamos que las clases están equilibradas (si una tuviera muchos más párrafos, el modelo tendería a predecirla) y
cómo de largos son los párrafos.

In [ ]:
df["n_palabras"] = df["text"].str.split().str.len()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
df["class"].value_counts().reindex(CLASES).plot.bar(ax=ax1, color="steelblue", rot=0)
ax1.set(title="Párrafos por clase", xlabel="", ylabel="párrafos")
for clase in CLASES:
    ax2.hist(df.loc[df["class"] == clase, "n_palabras"], bins=30, range=(0, 200), alpha=0.5, label=clase)
ax2.set(title="Longitud de los párrafos", xlabel="palabras", ylabel="párrafos")
ax2.legend()
plt.tight_layout()
plt.show()

df.groupby("class")["n_palabras"].describe()[["mean", "50%", "min", "max"]].round(0)

### 3.2 Lemas más frecuentes por clase

Los 15 lemas más frecuentes de cada clase (ya sin stop-words). Contamos en cuántos párrafos aparece cada lema, no
cuántas veces, para que un párrafo que repite mucho una palabra no pese de más.

In [ ]:
from collections import Counter

frecuentes = {clase: Counter(lema for lemas in df.loc[df["class"] == clase, "lemas"] for lema in set(lemas))
              for clase in CLASES}
pd.DataFrame({clase: [f"{lema} ({n})" for lema, n in frecuentes[clase].most_common(15)] for clase in CLASES})

In [ ]:
top = {clase: {lema for lema, _ in frecuentes[clase].most_common(50)} for clase in CLASES}
comunes = set.intersection(*top.values())
print(f"De los 50 lemas más frecuentes de cada clase, {len(comunes)} son comunes a todas:")
print(", ".join(sorted(comunes)))

Que muchos de los lemas más frecuentes sean comunes a todas las clases confirma que las temáticas son
**cercanas**: comparten el vocabulario general de la ciencia. Por eso conviene una representación como TF-IDF, que da
menos peso a las palabras que aparecen en todas partes.

## 4. Ruido en las etiquetas

Cada párrafo hereda la etiqueta del artículo del que sale, pero no todos los párrafos hablan del tema. Por ejemplo, un
artículo de física puede tener un párrafo sobre la historia de una universidad. Esos párrafos son **ruido**: llevan una
etiqueta que su contenido no justifica.

Para **medirlo** sin usar todavía ningún modelo, buscamos los **lemas distintivos** de cada clase: los que aparecen en
al menos `MIN_PARRAFOS` párrafos y, de esos párrafos, al menos un `UMBRAL` (70 %) son de una misma clase. Por ejemplo,
si `ĉelo` aparece en 40 párrafos y 36 son de `biologio`, es un lema distintivo de biología.

Después contamos, en cada párrafo, cuántos lemas distintivos tiene de su clase y cuántos de otra:

| Tipo | Condición | Qué significa |
|---|---|---|
| **temático** | más lemas de su clase que de cualquier otra | la etiqueta encaja con el contenido |
| **genérico** | ningún lema distintivo | no dice nada del tema (ruido) |
| **ambiguo** | empate entre su clase y otra | podría ser de las dos |
| **sospechoso** | más lemas de otra clase que de la suya | probablemente mal etiquetado (ruido) |

Para no hacer trampa, al estudiar un párrafo **se le quita de los recuentos**: un lema no puede ser distintivo solo
porque aparece en ese mismo párrafo.

In [ ]:
MIN_PARRAFOS = 5  # un lema tiene que salir en al menos 5 párrafos (sin contar el que se estudia)
UMBRAL = 0.7      # y el 70 % de ellos tienen que ser de la misma clase

# En cuántos párrafos aparece cada lema, en total y por clase
total = Counter()
por_clase = Counter()
for lemas_parrafo, clase in zip(df["lemas"], df["class"]):
    for lema in set(lemas_parrafo):
        total[lema] += 1
        por_clase[lema, clase] += 1


def clase_distintiva(lema, clase_propia):
    """Clase de la que es distintivo el lema sin contar el párrafo estudiado, o None."""
    n = total[lema] - 1
    if n < MIN_PARRAFOS:
        return None
    for clase in CLASES:
        apariciones = por_clase[lema, clase] - (clase == clase_propia)
        if apariciones / n >= UMBRAL:
            return clase
    return None


def tipo_de_parrafo(lemas_parrafo, clase_propia):
    votos = Counter(clase_distintiva(lema, clase_propia) for lema in set(lemas_parrafo))
    votos.pop(None, None)
    propios = votos.pop(clase_propia, 0)
    ajenos = max(votos.values(), default=0)
    if propios == ajenos == 0:
        return "genérico"
    if propios > ajenos:
        return "temático"
    return "ambiguo" if propios == ajenos else "sospechoso"


TIPOS = ["temático", "ambiguo", "genérico", "sospechoso"]
df["tipo"] = [tipo_de_parrafo(lemas_parrafo, clase) for lemas_parrafo, clase in zip(df["lemas"], df["class"])]

ruido = pd.crosstab(df["class"], df["tipo"], normalize="index").mul(100).reindex(columns=TIPOS).round(1)
ruido.loc["total"] = df["tipo"].value_counts(normalize=True).mul(100).reindex(TIPOS).round(1)
ruido

In [ ]:
ax = ruido.plot.barh(stacked=True, figsize=(10, 3.5), color=["seagreen", "gold", "darkgray", "indianred"])
ax.set(title="Tipo de párrafo según sus lemas distintivos", xlabel="% de párrafos", ylabel="")
ax.legend(bbox_to_anchor=(1, 1))
ax.invert_yaxis()
plt.tight_layout()
plt.show()

ruidosos = df["tipo"].isin(["genérico", "sospechoso"]).mean()
print(f"Ruido estimado (genéricos + sospechosos): {ruidosos:.1%} de los párrafos")

### 4.1 Ejemplos

Algunos párrafos de cada tipo de ruido, para comprobar a mano que la medida tiene sentido.

In [ ]:
pd.set_option("display.max_colwidth", 220)
for tipo in ["genérico", "sospechoso"]:
    print(f"--- {tipo} ---")
    display(df.loc[df["tipo"] == tipo, ["class", "text"]].sample(4, random_state=SEMILLA))

## 5. Modelo de clasificación

### 5.1 Partición entrenamiento / test

Separamos un 20 % del dataset como **test**. No se toca hasta la evaluación final, así la nota que obtenga el modelo
en test es una estimación honesta de cómo funcionará con textos nuevos. La partición es **estratificada** (cada clase
tiene la misma proporción en train y en test) y usa la semilla fija.

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, stratify=df["class"], random_state=SEMILLA)
print(f"Entrenamiento: {len(train)} párrafos · Test: {len(test)} párrafos")
pd.DataFrame({"train": train["class"].value_counts(), "test": test["class"].value_counts()})

### 5.2 Vectorización

Para entrenar hay que pasar cada párrafo a una fila de números (representación **tabular**). Probamos las dos técnicas
vistas en clase:

- **Bolsa de palabras (BoW):** una columna por lema; el valor es cuántas veces aparece en el párrafo.
- **TF-IDF:** igual, pero cada lema se pondera por lo raro que es en el corpus. Los lemas que salen en casi todos los
  párrafos (*povi*, *havi*, *granda*...) pesan poco, y los específicos (*ĉelo*, *elektrono*...) pesan mucho. Usamos
  `sublinear_tf` (logaritmo de la frecuencia) para que un lema repetido muchas veces no domine el párrafo.

En los dos casos la entrada ya son los lemas de `ciencianlp`, así que el vectorizador no vuelve a tokenizar
(`analyzer=identidad`). Con `min_df=2` se descartan los lemas que salen en un solo párrafo: no ayudan a generalizar y
suelen ser nombres propios o erratas.

El vectorizador va **dentro de un `Pipeline`** junto al clasificador. Así, en la validación cruzada el vocabulario y
los pesos IDF se calculan solo con los datos de entrenamiento de cada partición, sin mirar los de validación.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


def identidad(tokens):
    """Los párrafos ya llegan tokenizados y lematizados por ciencianlp."""
    return tokens


def tfidf():
    return TfidfVectorizer(analyzer=identidad, min_df=2, sublinear_tf=True)


matriz = tfidf().fit_transform(train["lemas"])
print(f"Matriz TF-IDF de entrenamiento: {matriz.shape[0]} párrafos x {matriz.shape[1]} lemas "
      f"({matriz.nnz / (matriz.shape[0] * matriz.shape[1]):.2%} de celdas distintas de cero)")

### 5.3 Comparación de modelos

Comparamos varias combinaciones con **validación cruzada estratificada de 5 particiones** sobre el conjunto de
entrenamiento, midiendo *accuracy* y **F1 macro** (la media del F1 de cada clase; trata igual a todas las clases).

- **Naive Bayes multinomial:** el clásico para texto, muy rápido.
- **Regresión logística:** lineal, da probabilidades bien calibradas, que necesitaremos para las páginas web.
- **SVM lineal:** suele ser de los mejores con texto, pero no da probabilidades.
- **Sin `ciencianlp`:** la misma regresión logística sobre las palabras tal cual, sin lematizar ni quitar
  stop-words. Sirve para medir cuánto aporta nuestra librería.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

# nombre: (columna de entrada, pipeline)
CANDIDATOS = {
    "BoW + Naive Bayes": ("lemas", make_pipeline(CountVectorizer(analyzer=identidad, min_df=2), MultinomialNB())),
    "TF-IDF + Naive Bayes": ("lemas", make_pipeline(tfidf(), MultinomialNB())),
    "TF-IDF + Regresión logística": ("lemas", make_pipeline(tfidf(), LogisticRegression(max_iter=2000))),
    "TF-IDF + SVM lineal": ("lemas", make_pipeline(tfidf(), LinearSVC(random_state=SEMILLA))),
    "Sin ciencianlp: TF-IDF palabras + Reg. logística": (
        "palabras", make_pipeline(tfidf(), LogisticRegression(max_iter=2000))),
}

filas = []
for nombre, (columna, pipeline) in CANDIDATOS.items():
    r = cross_validate(pipeline, train[columna], train["class"], cv=CV, scoring=["accuracy", "f1_macro"])
    filas.append({"modelo": nombre, "accuracy": r["test_accuracy"].mean(), "F1 macro": r["test_f1_macro"].mean(),
                  "F1 desv.": r["test_f1_macro"].std(), "probabilidades": hasattr(pipeline, "predict_proba")})
comparacion = pd.DataFrame(filas).set_index("modelo").sort_values("F1 macro", ascending=False)
comparacion.round(3)

In [ ]:
ax = comparacion["F1 macro"].plot.barh(xerr=comparacion["F1 desv."], color="steelblue", figsize=(9, 3.5))
ax.set(title="F1 macro en validación cruzada (5 particiones)", xlabel="F1 macro", ylabel="",
       xlim=(max(0, comparacion["F1 macro"].min() - 0.15), 1))
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**Qué vemos:**

- Todos los modelos con `ciencianlp` rondan un F1 macro de 0.83–0.86. Las diferencias entre ellos son del orden de
  su desviación típica, así que no hay un ganador claro: el problema está limitado más por los datos (y su ruido) que
  por el algoritmo.
- **Nuestra librería sí marca la diferencia:** el mismo modelo sobre palabras sin lematizar y con stop-words baja unos
  5 puntos de F1. Lematizar junta las formas de cada palabra (*ĉelo*, *ĉelojn*...) y quitar stop-words elimina ruido,
  algo especialmente útil con un dataset pequeño.

### 5.4 Ajuste del modelo elegido

Elegimos el mejor modelo **que dé probabilidades** (el enunciado pide la probabilidad de cada categoría para las
páginas web) y ajustamos su hiperparámetro principal con `GridSearchCV`, otra vez con validación cruzada sobre
entrenamiento:

- Regresión logística: `C` (cuanto más grande, menos regularización).
- Naive Bayes: `alpha` (suavizado de Laplace).

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV

REJILLAS = {
    LogisticRegression: {"logisticregression__C": [1, 3, 10, 30, 100]},
    MultinomialNB: {"multinomialnb__alpha": [0.01, 0.03, 0.1, 0.3, 1.0]},
}

elegido = comparacion[comparacion["probabilidades"]].index[0]
COLUMNA, base = CANDIDATOS[elegido]
busqueda = GridSearchCV(clone(base), REJILLAS[type(base[-1])], cv=CV, scoring="f1_macro")
busqueda.fit(train[COLUMNA], train["class"])
modelo = busqueda.best_estimator_  # ya reentrenado con todo el conjunto de entrenamiento

print("Modelo elegido:", elegido)
print("Mejor hiperparámetro:", busqueda.best_params_, f"(F1 macro en CV: {busqueda.best_score_:.3f})")
pd.DataFrame(busqueda.cv_results_)[["params", "mean_test_score", "std_test_score"]].round(3)

### 5.5 Evaluación en test

Ahora sí usamos el conjunto de test, una sola vez, con el modelo ya ajustado.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score

test = test.copy()
test["pred"] = modelo.predict(test[COLUMNA])
probas = modelo.predict_proba(test[COLUMNA])
test["confianza"] = probas.max(axis=1)

print(f"Accuracy: {accuracy_score(test['class'], test['pred']):.3f}")
print(f"F1 macro: {f1_score(test['class'], test['pred'], average='macro'):.3f}")
print()
print(classification_report(test["class"], test["pred"], digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_predictions(test["class"], test["pred"], labels=modelo.classes_, ax=axes[0],
                                        cmap="Blues", colorbar=False)
axes[0].set_title("Matriz de confusión (párrafos)")
ConfusionMatrixDisplay.from_predictions(test["class"], test["pred"], labels=modelo.classes_, ax=axes[1],
                                        cmap="Blues", colorbar=False, normalize="true", values_format=".0%")
axes[1].set_title("Matriz de confusión (% de cada clase real)")
plt.tight_layout()
plt.show()

**Qué vemos:** el resultado en test (≈ 0.87 de F1 macro) es parecido al de la validación cruzada, así que el
modelo no está sobreajustado. Los errores están repartidos entre todas las parejas de clases: no hay dos temáticas
que el modelo confunda sistemáticamente, lo que encaja con que las tres comparten vocabulario científico general.
Química es la clase con menos *recall*: parte de sus párrafos hablan de biología (bioquímica, genética) o de física
(estructura de la materia).

### 5.6 Análisis del modelo

#### Lemas más influyentes

Los lemas que más empujan al modelo hacia cada clase. En la regresión logística son los de mayor coeficiente; en Naive
Bayes, los que más se desvían de la media de las clases.

In [ ]:
vectorizador, clasificador = modelo[0], modelo[-1]
nombres = vectorizador.get_feature_names_out()
if hasattr(clasificador, "coef_"):
    pesos = clasificador.coef_
else:
    pesos = clasificador.feature_log_prob_ - clasificador.feature_log_prob_.mean(axis=0)

fig, axes = plt.subplots(1, len(modelo.classes_), figsize=(4 * len(modelo.classes_), 4))
for ax, clase, peso in zip(axes, modelo.classes_, pesos):
    mejores = np.argsort(peso)[-12:]
    ax.barh(nombres[mejores], peso[mejores], color="steelblue")
    ax.set_title(clase)
plt.suptitle("Lemas con más peso para cada clase")
plt.tight_layout()
plt.show()

**Qué vemos:** los lemas tienen sentido para cada temática (*bakterio*, *populacio*, *nervo* en biología;
*kvarko*, *tensio*, *relativeco* en física; *oksigeno*, *valento*, *reakcio* en química), así que el modelo aprende
vocabulario del tema y no casualidades del dataset.

También se ve una limitación de la librería: *ĥemio* y *kemio* son dos grafías de la misma palabra (la forma con
*ĥ* es la antigua) y quedan como lemas distintos. Normalizar *ĥ* → *k* en el lematizador las juntaría.

#### Confianza del modelo

Distribución de la probabilidad que da el modelo a la clase que predice, separando aciertos y errores. Si el modelo
está bien calibrado, los errores deberían concentrarse en confianzas bajas.

In [ ]:
test["acierto"] = test["pred"] == test["class"]
plt.figure(figsize=(8, 3.5))
plt.hist([test.loc[test["acierto"], "confianza"], test.loc[~test["acierto"], "confianza"]], bins=15,
         range=(1 / len(CLASES), 1), stacked=True, color=["seagreen", "indianred"], label=["aciertos", "errores"])
plt.xlabel("probabilidad de la clase predicha")
plt.ylabel("párrafos")
plt.legend()
plt.title("Confianza del modelo en test")
plt.show()

print(test.groupby("acierto")["confianza"].mean().rename({True: "aciertos", False: "errores"}).round(2))

#### Ejemplos de predicciones correctas e incorrectas

In [ ]:
columnas_proba = [f"P({clase})" for clase in modelo.classes_]
test[columnas_proba] = probas.round(2)
mostrar = ["class", "pred", *columnas_proba, "text"]

print("Aciertos con más confianza:")
display(test[test["acierto"]].nlargest(3, "confianza")[mostrar])
print("Errores con más confianza:")
display(test[~test["acierto"]].nlargest(6, "confianza")[mostrar])

**Qué vemos:** muchos de los "errores" más seguros no son fallos del modelo, sino **párrafos mal
etiquetados**: un párrafo de genética etiquetado como `kemio` o uno sobre la habitabilidad de los planetas en
`biologio`. El modelo acierta el tema real del párrafo, pero la etiqueta heredada del artículo es otra.

Aparece incluso una **referencia bibliográfica** ("Proceedings of the Royal Society...") que se coló en el dataset.
Es una mejora pendiente para `scripts/clean_dataset.py`.

Naive Bayes da probabilidades muy extremas (casi siempre 0 o 1), incluso cuando se equivoca. Es un defecto conocido
del modelo, que supone que las palabras son independientes. Por eso, para las páginas web no nos fijamos en un
párrafo suelto, sino en la media de todos los párrafos de la página.

#### ¿Los errores coinciden con el ruido?

En la sección 4 marcamos cada párrafo como temático, ambiguo, genérico o sospechoso **sin usar ningún modelo**. Si esa
medida tiene sentido, el modelo debería fallar sobre todo en los párrafos genéricos y sospechosos.

In [ ]:
errores_por_tipo = test.groupby("tipo")["acierto"].agg(párrafos="size", aciertos="mean").reindex(TIPOS)
errores_por_tipo["% errores"] = ((1 - errores_por_tipo["aciertos"]) * 100).round(1)
errores_por_tipo[["párrafos", "% errores"]]

**Qué vemos:** el modelo casi no falla en los párrafos temáticos y la mayoría de sus errores están en los
sospechosos y genéricos. Dos medidas independientes (lemas distintivos y modelo) señalan los mismos párrafos, lo que
confirma que **una buena parte del error que queda se debe al ruido de las etiquetas y no al modelo**.

### 5.7 Ruido según el modelo

Otra forma de medir el ruido: con **validación cruzada** sobre entrenamiento, cada párrafo se predice con un modelo
que no lo ha visto. Si el modelo le da a la clase de otro tema mucha probabilidad (≥ 60 %), es un párrafo
probablemente mal etiquetado.

Después probamos a **quitar esos párrafos del entrenamiento** y reentrenar. El test no se toca, así que la comparación
es justa (aunque el test también tiene ruido, que ningún modelo puede acertar).

In [ ]:
from sklearn.model_selection import cross_val_predict

probas_cv = cross_val_predict(clone(modelo), train[COLUMNA], train["class"], cv=CV, method="predict_proba")
pred_cv = modelo.classes_[probas_cv.argmax(axis=1)]
mal_etiquetado = (pred_cv != train["class"].to_numpy()) & (probas_cv.max(axis=1) >= 0.6)

print(f"Párrafos de entrenamiento probablemente mal etiquetados según el modelo: "
      f"{mal_etiquetado.sum()} ({mal_etiquetado.mean():.1%})")
print()
print("Comparación con la medida de la sección 4 (% de cada tipo que el modelo marca como mal etiquetado):")
print(pd.Series(mal_etiquetado, index=train.index).groupby(train["tipo"]).mean().reindex(TIPOS).mul(100).round(1))

In [ ]:
limpio = train[~mal_etiquetado]
modelo_limpio = clone(modelo).fit(limpio[COLUMNA], limpio["class"])
pred_limpio = modelo_limpio.predict(test[COLUMNA])

pd.DataFrame({
    "entrenamiento": [f"completo ({len(train)})", f"sin mal etiquetados ({len(limpio)})"],
    "accuracy": [accuracy_score(test["class"], test["pred"]), accuracy_score(test["class"], pred_limpio)],
    "F1 macro": [f1_score(test["class"], test["pred"], average="macro"),
                 f1_score(test["class"], pred_limpio, average="macro")],
}).set_index("entrenamiento").round(3)

**Qué vemos:** las dos medidas de ruido coinciden: el modelo marca como mal etiquetados sobre todo los
párrafos que la sección 4 llamó sospechosos o genéricos. Juntando las dos, estimamos que **entre un 13 % y un 19 % del
dataset tiene una etiqueta que su contenido no justifica**.

En cambio, quitar esos párrafos del entrenamiento **no mejora** el resultado en test, sino que lo empeora un poco. Hay
dos motivos: el test tiene el mismo tipo de ruido (el modelo "limpio" se aleja de esas etiquetas) y se pierde más de
un 10 % de los datos de entrenamiento. Por eso nos quedamos con el modelo entrenado con todo el conjunto. Para mejorar
de verdad habría que reducir el ruido en origen, al construir el dataset (por ejemplo, descartando los párrafos sin
lemas distintivos).

## 6. Clasificación de páginas web

### 6.1 Clasificar un texto nuevo

Para clasificar cualquier texto se aplica exactamente el mismo preprocesado que al entrenar: `ciencianlp` lo
tokeniza y lematiza, nos quedamos con los lemas útiles y el modelo devuelve la probabilidad de cada clase.

In [ ]:
def clasificar(textos):
    """Probabilidad de cada clase para una lista de textos (una fila por texto)."""
    if COLUMNA == "lemas":
        entrada = [lemas_utiles(nlp(texto)) for texto in textos]
    else:
        entrada = [[t.text.lower() for t in nlp(texto) if not t.is_punct] for texto in textos]
    return pd.DataFrame(modelo.predict_proba(entrada), columns=modelo.classes_)


frases = [
    "La elektronoj moviĝas ĉirkaŭ la atomkerno pro la elektromagneta forto.",
    "La ĉeloj de la plantoj enhavas kloroplastojn kaj DNA.",
    "Kiam oni miksas acidon kun bazo, okazas kemia reakcio kaj formiĝas salo kaj akvo.",
]
clasificar(frases).round(3).assign(texto=frases)

### 6.2 Extracción del texto de una página web (*web scraping*)

Descargamos el HTML con `requests` y extraemos el texto con `BeautifulSoup`:

1. Nos quedamos con la zona del contenido principal (`div.mw-parser-output` en Wikipedia, `<article>` en otras webs y,
   si no hay ninguna, todo el `<body>`). Así se evitan menús, cabeceras y pies de página.
2. Quitamos lo que no es texto del artículo: referencias (`[1]`), tablas, fórmulas, scripts...
3. Cogemos los párrafos (`<p>`) y los limpiamos igual que el dataset: sin referencias, enlaces ni `;`. Descartamos los
   de menos de `MIN_PALABRAS_WEB` palabras, que suelen ser pies de foto o frases sueltas.

**Robustez:** la sesión reintenta sola si el servidor falla (errores 429 y 5xx) y cualquier otro problema (sin red,
página que no existe, HTML raro...) se captura: la función avisa y devuelve una lista vacía, así el cuaderno sigue.

In [ ]:
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

MIN_PALABRAS_WEB = 20

SESION = requests.Session()
SESION.headers["User-Agent"] = f"ciencianlp-practica/1.0 (https://github.com/{REPO}; proyecto educativo)"
SESION.mount("https://", HTTPAdapter(max_retries=Retry(total=3, backoff_factor=1,
                                                         status_forcelist=(429, 500, 502, 503, 504))))


def limpiar(texto):
    """Quita referencias [1], enlaces, espacios de más y el carácter ';' (como en el dataset)."""
    texto = re.sub(r"\[\s*[\w\s]{1,10}\]", " ", texto)  # [1], [ 2 ], [noto 3]
    texto = re.sub(r"https?://\S+|www\.\S+", " ", texto)
    texto = texto.replace(";", ",")
    texto = re.sub(r"\s+([,.:)])", r"\1", texto)        # "palabra ," -> "palabra,"
    return re.sub(r"\s+", " ", texto).strip()


def extraer_parrafos(url):
    """Párrafos limpios de una página web. Si algo falla, avisa y devuelve una lista vacía."""
    try:
        respuesta = SESION.get(url, timeout=20)
        respuesta.raise_for_status()
        sopa = BeautifulSoup(respuesta.text, "html.parser")
        zona = sopa.select_one("div.mw-parser-output") or sopa.find("article") or sopa.body or sopa
        for basura in zona.select("sup.reference, table, style, script, figure, .mw-editsection, .mwe-math-element"):
            basura.decompose()
        parrafos = [limpiar(p.get_text(" ", strip=True)) for p in zona.find_all("p")]
        # Solo cuentan las palabras de 3 letras o más, para descartar fórmulas sueltas como "C 6 H 12 O 6"
        return [p for p in parrafos if len(re.findall(r"[^\W\d_]{3,}", p)) >= MIN_PALABRAS_WEB]
    except Exception as error:  # sin red, 404, timeout, HTML inesperado...
        print(f"No se pudo leer {url}: {type(error).__name__}: {error}")
        return []

### 6.3 Páginas elegidas

Elegimos artículos de la Wikipedia en esperanto de cada temática. Para que la prueba sea justa, ninguno de ellos se
usó para construir el dataset (lo comprobamos abajo). De biología ponemos dos: *Ekologio*, claramente biológico, y
*Fotosintezo*, un tema de frontera entre biología y química (bioquímica). Además añadimos:

- **Una página de control fuera de tema** (fútbol): el modelo solo conoce tres clases y siempre elige una, así que
  esperamos probabilidades más repartidas.
- **Una URL que no existe**, para comprobar que el cuaderno sobrevive a una web caída.

Para usar el cuaderno con otro dataset basta con cambiar esta lista.

In [ ]:
PAGINAS = {
    "Termodinamiko (física)": "https://eo.wikipedia.org/wiki/Termodinamiko",
    "Ekologio (biología)": "https://eo.wikipedia.org/wiki/Ekologio",
    "Fotosintezo (biología, frontera con química)": "https://eo.wikipedia.org/wiki/Fotosintezo",
    "Organika kemio (química)": "https://eo.wikipedia.org/wiki/Organika_kemio",
    "Futbalo (control, fuera de tema)": "https://eo.wikipedia.org/wiki/Futbalo",
    "Web caída (prueba de robustez)": "https://esta-web-no-existe.invalid/paĝo",
}

parrafos_web = {nombre: extraer_parrafos(url) for nombre, url in PAGINAS.items()}
parrafos_web = {nombre: parrafos for nombre, parrafos in parrafos_web.items() if parrafos}  # quita las que fallaron
for nombre, parrafos in parrafos_web.items():
    print(f"{nombre}: {len(parrafos)} párrafos")

In [ ]:
# ¿Algún párrafo de las páginas está en el dataset? Buscamos un trozo de 8 palabras de cada uno en el corpus.
corpus = " ".join(df["text"])
for nombre, parrafos in parrafos_web.items():
    repetidos = sum(" ".join(p.split()[2:10]) in corpus for p in parrafos)
    print(f"{nombre}: {repetidos} de {len(parrafos)} párrafos aparecen en el dataset")

### 6.4 Resultados

Clasificamos **cada párrafo** por separado y calculamos la probabilidad de la página como la **media** de las
probabilidades de sus párrafos. Lo hacemos así, y no clasificando todo el texto de golpe, porque:

- el modelo se entrenó con párrafos, así que funciona mejor con textos de ese tamaño;
- con un texto muy largo, Naive Bayes da probabilidades casi siempre de 0 y 1. La media de los párrafos refleja mejor
  cuánta parte de la página habla de cada tema.

In [ ]:
if not parrafos_web:
    print("No se pudo descargar ninguna página: se omite la clasificación web.")
    resultados = pd.DataFrame(columns=list(modelo.classes_))
else:
    detalle = pd.concat([clasificar(parrafos).assign(página=nombre, texto=parrafos)
                         for nombre, parrafos in parrafos_web.items()], ignore_index=True)
    detalle["predicción"] = detalle[list(modelo.classes_)].idxmax(axis=1)

    resultados = detalle.groupby("página", sort=False)[list(modelo.classes_)].mean()
    resultados["predicción"] = resultados.idxmax(axis=1)
    resultados["párrafos"] = detalle.groupby("página", sort=False).size()
resultados.round(3)

In [ ]:
if len(resultados):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

    resultados[list(modelo.classes_)].plot.barh(stacked=True, ax=ax1, legend=False)
    ax1.set(title="Probabilidad media de cada clase", xlabel="probabilidad", ylabel="", xlim=(0, 1))

    reparto = pd.crosstab(detalle["página"], detalle["predicción"], normalize="index")
    reparto = reparto.reindex(index=resultados.index, columns=modelo.classes_, fill_value=0)
    reparto.plot.barh(stacked=True, ax=ax2)
    ax2.set(title="% de párrafos asignados a cada clase", xlabel="% de párrafos", xlim=(0, 1))
    ax2.legend(bbox_to_anchor=(1, 1))

    ax1.invert_yaxis()
    plt.tight_layout()
    plt.show()

Para ver de dónde salen esas medias, estos son los párrafos de cada página en los que el modelo **duda más**
(la clase más probable tiene menos probabilidad):

In [ ]:
if len(resultados):
    detalle["confianza"] = detalle[list(modelo.classes_)].max(axis=1)
    for nombre in resultados.index:
        print(f"--- {nombre} ---")
        display(detalle[detalle["página"] == nombre].nsmallest(2, "confianza")
                [[*modelo.classes_, "predicción", "texto"]].round(2))

### 6.5 Análisis de los resultados web

- **Ekologio** (≈ 96 % biología), **Organika kemio** (≈ 90 % química) y **Termodinamiko** (≈ 64 % física) se clasifican
  bien. La termodinámica tiene bastante probabilidad de química (≈ 27 %), y tiene sentido: habla de calor, energía,
  presión y gases, que también son conceptos de química física.
- **Fotosintezo** sale como **química** (≈ 87 %). No es un error absurdo: el artículo explica la fotosíntesis a nivel
  molecular (CO₂, glucosa, clorofila, reacciones de oxidación...), que es **bioquímica**. Es justo el tipo de texto de
  frontera que hace que las temáticas sean cercanas. Los párrafos más "biológicos" (los cloroplastos, las plantas)
  sí van a biología.
- La página de **fútbol** sale como biología (≈ 71 %). El modelo **siempre** elige una de las tres clases que
  conoce, aunque el texto no sea de ninguna. Probablemente el vocabulario sobre personas y su cuerpo (*ludanto*,
  *homo*, *korpo*...) se parece más al de biología. Para detectar textos fuera de tema haría falta una cuarta clase
  "otros" en el dataset, o rechazar las páginas cuya probabilidad máxima no supere un umbral.
- La **web caída** no detiene el cuaderno: se avisa del error y se sigue con el resto de páginas.
- Mirar los **párrafos dudosos** ayuda a interpretar las medias. En *Organika kemio*, por ejemplo, el párrafo que
  más baja la probabilidad de química habla del clima, no de química.